In [1]:
import os
import cv2
import argparse
import torch
import torchvision.transforms as T

from repnet import utils, plots
from repnet.model import RepNet


In [2]:
video_path = './event_video/class3/user02_fluorescent.mp4' # './videos/cheetah_running_at_63_mph_102_kph.mp4' # path to video file

In [3]:
# repnet model variables 
weights = './pytorch_weights.pth'
device = 'cuda'
strides = [1, 2, 3, 4, 8]
fps = 60 

OUT_VISUALIZATIONS_DIR = './visualization/'

In [4]:
# read frames and apply preprocessing 
transform = T.Compose([
    T.ToPILImage(),
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=0.5, std=0.5),      
])

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
raw_frames, frames = [], []

# frames_count = 600
while cap.isOpened() :
    ret, frame = cap.read()
    if not ret or frame is None:
        break
    raw_frames.append(frame)
    frame = transform(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frames.append(frame)
    # if not frames_count:
    #     break
    # frames_count -= 1
cap.release()

In [5]:
len(frames)

312

In [6]:
# Load model
model = RepNet()
state_dict = torch.load(weights)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

Using cache found in /home/rlwagun/.cache/torch/hub/huggingface_pytorch-image-models_main
/tmp/ipykernel_3663421/542158605.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


RepNet(
  (encoder): ResNetV2(
    (stem): Sequential(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (pool): Sequential(
        (0): ZeroPad2d((1, 1, 1, 1))
        (1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (stages): Sequential(
      (0): ResNetStage(
        (blocks): Sequential(
          (0): PreActBottleneck(
            (downsample): DownsampleConv(
              (conv): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
              (norm): Identity()
            )
            (norm1): BatchNormAct2d(
              64, eps=1.001e-05, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): ReLU(inplace=True)
            )
            (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (norm2): BatchNormAct2d(
              64, eps=1.001e-05, momentum=0.1, affine=True, track_running_stats=True
            

In [7]:
import torch
import psutil

# GPU Memory Usage
gpu_memory = torch.cuda.memory_allocated() / (1024 * 1024)  # Convert to MB
print(f"GPU Memory Usage: {gpu_memory:.2f} MB")

# CPU RAM Usage
ram_usage = psutil.Process().memory_info().rss / (1024 * 1024)  # Convert to MB
print(f"CPU RAM Usage: {ram_usage:.2f} MB")

def get_model_size(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())  # Model parameters
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())  # Model buffers (like BatchNorm stats)
    total_size_mb = (param_size + buffer_size) / (1024 * 1024)  # Convert to MB
    return total_size_mb

model_size = get_model_size(model)
print(f"Model Size: {model_size:.2f} MB")

GPU Memory Usage: 98.27 MB
CPU RAM Usage: 1011.46 MB
Model Size: 98.00 MB


In [9]:
%%timeit
# Test multiple strides and pick the best one
print('Running inference on multiple stride values...')
best_stride, best_confidence, best_period_length, best_period_count, best_periodicity_score, best_embeddings = None, None, None, None, None, None
for stride in strides:
    # Apply stride
    stride_frames = frames[::stride]
    stride_frames = stride_frames[:(len(stride_frames) // 64) * 64]
    if len(stride_frames) < 64:
        continue # Skip this stride if there are not enough frames
    stride_frames = torch.stack(stride_frames, axis=0).unflatten(0, (-1, 64)).movedim(1, 2) # Convert to N x C x D x H x W
    stride_frames = stride_frames.to(device)
    # Run inference
    raw_period_length, raw_periodicity_score, embeddings = [], [], []
    with torch.no_grad():
        for i in range(stride_frames.shape[0]):  # Process each batch separately to avoid OOM
            batch_period_length, batch_periodicity, batch_embeddings = model(stride_frames[i].unsqueeze(0))
            raw_period_length.append(batch_period_length[0].cpu())
            raw_periodicity_score.append(batch_periodicity[0].cpu())
            embeddings.append(batch_embeddings[0].cpu())
    # Post-process results
    raw_period_length, raw_periodicity_score, embeddings = torch.cat(raw_period_length), torch.cat(raw_periodicity_score), torch.cat(embeddings)
    confidence, period_length, period_count, periodicity_score = model.get_counts(raw_period_length, raw_periodicity_score, stride)
    if best_confidence is None or confidence > best_confidence:
        best_stride, best_confidence, best_period_length, best_period_count, best_periodicity_score, best_embeddings = stride, confidence, period_length, period_count, periodicity_score, embeddings
if best_stride is None:
    raise RuntimeError('The stride values used are too large and nove 64 video chunk could be sampled. Try different values for --strides.')
print(f'Predicted a period length of {best_period_length/fps:.1f} seconds (~{int(best_period_length)} frames) with a confidence of {best_confidence:.2f} using a stride of {best_stride} frames.')

# Generate plots and videos
# print(f'Save plots and video with counts to {OUT_VISUALIZATIONS_DIR}...')
# os.makedirs(OUT_VISUALIZATIONS_DIR, exist_ok=True)
# dist = torch.cdist(best_embeddings, best_embeddings, p=2)**2
# tsm_img = plots.plot_heatmap(dist.numpy(), log_scale=True)
# pca_img = plots.plot_pca(best_embeddings.numpy())
# cv2.imwrite(os.path.join(OUT_VISUALIZATIONS_DIR, 'tsm.png'), tsm_img)
# cv2.imwrite(os.path.join(OUT_VISUALIZATIONS_DIR, 'pca.png'), pca_img)


Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period length of 0.4 seconds (~23 frames) with a confidence of 0.98 using a stride of 4 frames.
Running inference on multiple stride values...
Predicted a period leng

In [43]:
# Generate video with counts
rep_frames = plots.plot_repetitions(raw_frames[:len(best_period_count)], best_period_count.tolist(), best_periodicity_score.tolist() if not True else None)
video = cv2.VideoWriter(os.path.join(OUT_VISUALIZATIONS_DIR, 'repetitions.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), fps, rep_frames[0].shape[:2][::-1])
for frame in rep_frames:
    video.write(frame)
video.release()

In [44]:
best_period_count.tolist()[-1]
round(best_period_count.tolist()[-1])

11

In [8]:
import torch
import psutil

# GPU Memory Usage
gpu_memory = torch.cuda.memory_allocated() / (1024 * 1024)  # Convert to MB
print(f"GPU Memory Usage: {gpu_memory:.2f} MB")

# CPU RAM Usage
ram_usage = psutil.Process().memory_info().rss / (1024 * 1024)  # Convert to MB
print(f"CPU RAM Usage: {ram_usage:.2f} MB")

GPU Memory Usage: 106.40 MB
CPU RAM Usage: 1434.41 MB
